# 🚀 GPU-Optimized ARIMA vs LSTM for Time Series Forecasting

## Key Optimizations:
✅ TensorFlow GPU acceleration for LSTM  
✅ CuPy for GPU-based numerical operations  
✅ Optimized memory management  
✅ Batch processing to prevent memory overflow  
✅ Lightweight ARIMA implementation  
✅ Mixed precision training for faster computation

## 1. GPU Setup and Verification

In [ ]:
# =============================================================================
# GPU CONFIGURATION AND VERIFICATION
# =============================================================================

import os
import sys
import gc
import warnings
warnings.filterwarnings('ignore')

# Force TensorFlow to use GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Use first GPU
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import TensorFlow and configure GPU
import tensorflow as tf

print("🔍 Checking GPU availability...")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"Num GPUs: {len(tf.config.list_physical_devices('GPU'))}")

# Configure GPU memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU memory growth configured")
        
        # Optional: Limit GPU memory
        # tf.config.set_logical_device_configuration(
        #     gpus[0],
        #     [tf.config.LogicalDeviceConfiguration(memory_limit=4096)]  # Limit to 4GB
        # )
    except RuntimeError as e:
        print(f"❌ GPU configuration error: {e}")
else:
    print("⚠️ No GPU found. The script will still work but slower.")
    print("   To use GPU, ensure CUDA and cuDNN are installed.")

# Try to import CuPy for GPU-accelerated NumPy operations
USE_CUPY = False
try:
    import cupy as cp
    USE_CUPY = True
    print(f"✅ CuPy available for GPU acceleration")
    print(f"   CuPy version: {cp.__version__}")
    
    # Set memory pool for efficient allocation
    mempool = cp.get_default_memory_pool()
    mempool.set_limit(size=2 * 1024**3)  # Limit to 2GB
except ImportError:
    print("⚠️ CuPy not installed. Using NumPy (CPU) for array operations")
    print("   Install with: pip install cupy-cuda11x  # Replace 11x with your CUDA version")
    cp = np  # Fallback to NumPy

# Enable mixed precision for faster GPU computation
from tensorflow.keras import mixed_precision
if gpus:
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("✅ Mixed precision enabled for faster GPU training")
    print(f"   Compute dtype: {policy.compute_dtype}")
    print(f"   Variable dtype: {policy.variable_dtype}")

## 2. Optimized Data Loading with Memory Management

In [ ]:
# =============================================================================
# MEMORY-EFFICIENT DATA LOADING
# =============================================================================

def load_data_optimized(file_path, target_col, chunk_size=10000):
    """
    Load data in chunks to prevent memory overflow
    """
    try:
        # Try loading entire file first
        df = pd.read_csv(file_path, parse_dates=['Date and Time'], index_col='Date and Time')
        print(f"✅ Data loaded: {len(df)} rows")
        
        # Convert to float32 to save memory
        for col in df.select_dtypes(include=[np.float64]).columns:
            df[col] = df[col].astype(np.float32)
        
        # Extract target series
        series = df[target_col].copy()
        
        # Clear dataframe from memory
        del df
        gc.collect()
        
        return series
        
    except MemoryError:
        print("⚠️ Memory error - loading in chunks...")
        
        # Load in chunks
        chunks = []
        for chunk in pd.read_csv(file_path, chunksize=chunk_size, 
                                 parse_dates=['Date and Time'], 
                                 index_col='Date and Time'):
            chunks.append(chunk[target_col])
        
        series = pd.concat(chunks)
        del chunks
        gc.collect()
        
        return series

# Load data
print("📊 Loading data with memory optimization...")
target_col = 'Hourly Demand Met (in MW)'

try:
    series = load_data_optimized(os.path.join(os.environ.get('DATA_DIR', '../data'), 'final_4year.csv'), target_col)
except FileNotFoundError:
    try:
        series = load_data_optimized('final_4year.csv', target_col)
    except:
        print("❌ Data file not found. Please check the path.")
        raise

# Data preprocessing
series = series.sort_index()
series = series.interpolate(method='linear', limit=24).ffill().bfill()

# Ensure hourly frequency
full_range = pd.date_range(start=series.index.min(), end=series.index.max(), freq='h')
series = series.reindex(full_range).interpolate(method='linear').ffill().bfill()

print(f"📈 Series loaded: {len(series):,} samples")
print(f"   Memory usage: {series.memory_usage(deep=True) / 1024**2:.2f} MB")
print(f"   Date range: {series.index.min()} to {series.index.max()}")

## 3. Train/Test Split with GPU Arrays

In [ ]:
# =============================================================================
# TRAIN/TEST SPLIT WITH GPU OPTIMIZATION
# =============================================================================

split_ratio = 0.90
split_idx = int(len(series) * split_ratio)

train = series.iloc[:split_idx]
test = series.iloc[split_idx:]

print(f"📊 Data Split:")
print(f"   Training: {len(train):,} samples ({len(train)/len(series)*100:.1f}%)")
print(f"   Testing: {len(test):,} samples ({len(test)/len(series)*100:.1f}%)")

# Convert to GPU arrays if CuPy is available
if USE_CUPY:
    print("\n🚀 Transferring data to GPU...")
    train_gpu = cp.asarray(train.values, dtype=cp.float32)
    test_gpu = cp.asarray(test.values, dtype=cp.float32)
    print("   ✅ Data transferred to GPU memory")
else:
    train_gpu = train.values.astype(np.float32)
    test_gpu = test.values.astype(np.float32)

## 4. Lightweight ARIMA Implementation

In [ ]:
# =============================================================================
# LIGHTWEIGHT ARIMA WITH REDUCED MEMORY FOOTPRINT
# =============================================================================

from statsmodels.tsa.statespace.sarimax import SARIMAX
import json

print("🔍 Fitting lightweight ARIMA model...\n")

# Use simple parameters to reduce computation
# Note: ARIMA cannot run on GPU, so we optimize for memory and speed

class LightweightARIMA:
    def __init__(self):
        self.order = (1, 1, 1)
        self.seasonal_order = (1, 1, 1, 24)
        self.model = None
        self.fitted = None
        
    def fit(self, train_data, sample_size=5000):
        """
        Fit ARIMA on a sample to reduce memory usage
        """
        print(f"📊 Fitting ARIMA on {min(sample_size, len(train_data))} samples...")
        
        # Use only recent data if dataset is large
        if len(train_data) > sample_size:
            train_sample = train_data[-sample_size:]
            print(f"   Using last {sample_size} samples to reduce memory")
        else:
            train_sample = train_data
        
        try:
            # Fit simple SARIMAX model
            self.model = SARIMAX(
                train_sample,
                order=self.order,
                seasonal_order=self.seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
                concentrate_scale=True,  # Reduces memory usage
                use_exact_diffuse=False  # Faster computation
            )
            
            # Fit with limited iterations
            self.fitted = self.model.fit(
                disp=False,
                maxiter=50,  # Limit iterations
                method='powell',  # Faster than default
                low_memory=True  # Use low memory mode
            )
            
            print(f"   ✅ ARIMA fitted successfully")
            print(f"   AIC: {self.fitted.aic:.2f}")
            
        except Exception as e:
            print(f"   ❌ ARIMA failed, using naive forecast: {str(e)}")
            self.fitted = None
            
    def predict(self, n_periods):
        """
        Generate predictions
        """
        if self.fitted is not None:
            return self.fitted.forecast(steps=n_periods)
        else:
            # Naive forecast - return last value
            return np.full(n_periods, train.iloc[-1])

# Fit lightweight ARIMA
arima_light = LightweightARIMA()
arima_light.fit(train, sample_size=5000)  # Use only last 5000 samples

# Generate predictions
print("\n📈 Generating ARIMA predictions...")
arima_predictions = arima_light.predict(len(test))
arima_pred_series = pd.Series(arima_predictions, index=test.index)

# Handle NaN values
if arima_pred_series.isnull().any():
    arima_pred_series = arima_pred_series.ffill().bfill()
    if arima_pred_series.isnull().any():
        arima_pred_series = arima_pred_series.fillna(train.mean())

print(f"   Generated {len(arima_pred_series)} predictions")

# Clear memory
del arima_light.model
gc.collect()

## 5. GPU-Accelerated LSTM Model

In [ ]:
# =============================================================================
# GPU-OPTIMIZED DATA PREPARATION FOR LSTM
# =============================================================================

from sklearn.preprocessing import MinMaxScaler

# Use GPU for scaling if available
scaler = MinMaxScaler(feature_range=(0, 1))

# Prepare data
train_scaled = scaler.fit_transform(train.values.reshape(-1, 1)).astype(np.float32)
test_scaled = scaler.transform(test.values.reshape(-1, 1)).astype(np.float32)

# Create sequences with optimized memory usage
LOOKBACK = 60
BATCH_SIZE = 256  # Larger batch size for GPU efficiency

def create_sequences_gpu(data, lookback, batch_size=None):
    """
    Create sequences optimized for GPU training
    """
    n_samples = len(data) - lookback
    
    # Pre-allocate arrays
    X = np.zeros((n_samples, lookback, 1), dtype=np.float32)
    y = np.zeros((n_samples,), dtype=np.float32)
    
    for i in range(n_samples):
        X[i] = data[i:i + lookback]
        y[i] = data[i + lookback, 0]
    
    # Ensure batch alignment for GPU
    if batch_size:
        n_batches = n_samples // batch_size
        n_samples = n_batches * batch_size
        X = X[:n_samples]
        y = y[:n_samples]
    
    return X, y

print("📊 Creating LSTM sequences...")
X_train, y_train = create_sequences_gpu(train_scaled, LOOKBACK, BATCH_SIZE)

print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   Memory usage: {(X_train.nbytes + y_train.nbytes) / 1024**2:.2f} MB")

In [ ]:
# =============================================================================
# BUILD GPU-OPTIMIZED LSTM MODEL
# =============================================================================

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Clear any existing models
tf.keras.backend.clear_session()

# Build model with GPU optimization
with tf.device('/GPU:0'):
    lstm_model = Sequential([
        # CuDNN-optimized LSTM (automatically used when GPU available)
        LSTM(units=64,  # Increased units for GPU
             return_sequences=True,
             input_shape=(LOOKBACK, 1),
             activation='tanh',  # Required for CuDNN
             recurrent_activation='sigmoid'),  # Required for CuDNN
        
        Dropout(0.2),  # Regularization
        
        LSTM(units=32,
             return_sequences=False,
             activation='tanh',
             recurrent_activation='sigmoid'),
        
        Dropout(0.2),
        
        Dense(units=16, activation='relu'),
        
        Dense(units=1, dtype='float32')  # Output in float32 for mixed precision
    ])

# Compile with optimized settings
optimizer = Adam(learning_rate=0.001)

lstm_model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=['mae']
)

print("🏗️ LSTM Model Architecture (GPU-Optimized):")
lstm_model.summary()

# Verify GPU placement
if tf.config.list_physical_devices('GPU'):
    print("\n✅ Model will run on GPU")
else:
    print("\n⚠️ Model will run on CPU (slower)")

In [ ]:
# =============================================================================
# TRAIN LSTM ON GPU WITH OPTIMIZED SETTINGS
# =============================================================================

# Callbacks for better training
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("🚀 Training LSTM on GPU...\n")

# Create TensorFlow dataset for efficient GPU loading
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Validation split
val_split = 0.1
val_size = int(len(X_train) * val_split)
val_dataset = train_dataset.take(val_size // BATCH_SIZE)
train_dataset = train_dataset.skip(val_size // BATCH_SIZE)

# Train with mixed precision and GPU
with tf.device('/GPU:0'):
    history = lstm_model.fit(
        train_dataset,
        epochs=30,  # More epochs since GPU is fast
        validation_data=val_dataset,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

print(f"\n✅ Training complete!")
print(f"   Best validation loss: {min(history.history['val_loss']):.6f}")
print(f"   Epochs trained: {len(history.history['loss'])}")

# Clear GPU memory
tf.keras.backend.clear_session()
gc.collect()

In [ ]:
# =============================================================================
# GENERATE LSTM PREDICTIONS ON GPU
# =============================================================================

print("📈 Generating LSTM predictions on GPU...")

# Prepare test sequences
full_scaled = np.concatenate([train_scaled, test_scaled])
test_start_idx = len(train_scaled)

# Batch prediction for GPU efficiency
PRED_BATCH_SIZE = 512

def predict_batch_gpu(model, sequences):
    """
    Batch prediction on GPU
    """
    n_samples = len(sequences)
    predictions = np.zeros(n_samples, dtype=np.float32)
    
    for i in range(0, n_samples, PRED_BATCH_SIZE):
        batch_end = min(i + PRED_BATCH_SIZE, n_samples)
        batch = sequences[i:batch_end]
        
        with tf.device('/GPU:0'):
            batch_pred = model.predict(batch, verbose=0, batch_size=PRED_BATCH_SIZE)
        
        predictions[i:batch_end] = batch_pred.flatten()
    
    return predictions

# Create test sequences
test_sequences = []
for i in range(test_start_idx, len(full_scaled)):
    seq = full_scaled[i - LOOKBACK:i, 0].reshape((1, LOOKBACK, 1))
    test_sequences.append(seq)

test_sequences = np.vstack(test_sequences)

# Predict in batches on GPU
lstm_predictions_scaled = predict_batch_gpu(lstm_model, test_sequences)

# Inverse transform
lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled.reshape(-1, 1)).flatten()
lstm_pred_series = pd.Series(lstm_predictions, index=test.index)

print(f"\n✅ Generated {len(lstm_pred_series):,} predictions")
print(f"   Prediction range: [{lstm_pred_series.min():.2f}, {lstm_pred_series.max():.2f}]")

## 6. Evaluation with GPU Arrays

In [ ]:
# =============================================================================
# GPU-ACCELERATED METRICS CALCULATION
# =============================================================================

def calculate_metrics_gpu(y_true, y_pred, model_name):
    """
    Calculate metrics using GPU if available
    """
    if USE_CUPY:
        # Use CuPy for GPU computation
        y_true_gpu = cp.asarray(y_true, dtype=cp.float32)
        y_pred_gpu = cp.asarray(y_pred, dtype=cp.float32)
        
        # Remove NaN values
        valid_mask = ~(cp.isnan(y_true_gpu) | cp.isnan(y_pred_gpu))
        y_true_clean = y_true_gpu[valid_mask]
        y_pred_clean = y_pred_gpu[valid_mask]
        
        # Calculate metrics on GPU
        mse = cp.mean((y_true_clean - y_pred_clean) ** 2)
        rmse = cp.sqrt(mse)
        mae = cp.mean(cp.abs(y_true_clean - y_pred_clean))
        
        # Transfer back to CPU
        rmse = float(rmse.get())
        mae = float(mae.get())
        valid_count = int(valid_mask.sum().get())
        
    else:
        # CPU fallback
        y_true_arr = np.array(y_true, dtype=np.float32)
        y_pred_arr = np.array(y_pred, dtype=np.float32)
        
        valid_mask = ~(np.isnan(y_true_arr) | np.isnan(y_pred_arr))
        y_true_clean = y_true_arr[valid_mask]
        y_pred_clean = y_pred_arr[valid_mask]
        
        rmse = np.sqrt(np.mean((y_true_clean - y_pred_clean) ** 2))
        mae = np.mean(np.abs(y_true_clean - y_pred_clean))
        valid_count = valid_mask.sum()
    
    return {
        'Model': model_name,
        'RMSE (MW)': round(rmse, 2),
        'MAE (MW)': round(mae, 2),
        'Valid Samples': f"{valid_count}/{len(y_true)}"
    }

# Calculate metrics
print("📊 Calculating metrics...")

arima_metrics = calculate_metrics_gpu(test.values, arima_pred_series.values, 'ARIMA (CPU)')
lstm_metrics = calculate_metrics_gpu(test.values, lstm_pred_series.values, 'LSTM (GPU)')

# Display results
results_df = pd.DataFrame([arima_metrics, lstm_metrics])

print("\n" + "="*60)
print("📊 MODEL COMPARISON RESULTS")
print("="*60)
print(f"\n{results_df.to_string(index=False)}")

# Determine winner
if arima_metrics['RMSE (MW)'] < lstm_metrics['RMSE (MW)']:
    print(f"\n🏆 Winner: ARIMA (by {lstm_metrics['RMSE (MW)'] - arima_metrics['RMSE (MW)']:.2f} MW)")
else:
    print(f"\n🏆 Winner: LSTM (by {arima_metrics['RMSE (MW)'] - lstm_metrics['RMSE (MW)']:.2f} MW)")

## 7. Visualization

In [ ]:
# =============================================================================
# VISUALIZATION
# =============================================================================

# Get last 7 days for visualization
last_7_days = min(24 * 7, len(test))
actual_7d = test.iloc[-last_7_days:]
arima_7d = arima_pred_series.iloc[-last_7_days:]
lstm_7d = lstm_pred_series.iloc[-last_7_days:]

fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(actual_7d.index, actual_7d.values, 
        label='Actual', color='#2c3e50', linewidth=2.5, alpha=0.9)
ax.plot(arima_7d.index, arima_7d.values, 
        label=f'ARIMA (RMSE: {arima_metrics["RMSE (MW)"]} MW)', 
        color='#e74c3c', linewidth=2, linestyle='--', alpha=0.8)
ax.plot(lstm_7d.index, lstm_7d.values, 
        label=f'LSTM-GPU (RMSE: {lstm_metrics["RMSE (MW)"]} MW)', 
        color='#27ae60', linewidth=2, linestyle='-.', alpha=0.8)

ax.set_title('ARIMA vs GPU-LSTM: Last 7 Days of Test Data', fontsize=16, fontweight='bold')
ax.set_xlabel('Date & Time', fontsize=12)
ax.set_ylabel('Hourly Demand Met (MW)', fontsize=12)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Memory Cleanup

In [ ]:
# =============================================================================
# MEMORY CLEANUP
# =============================================================================

print("🧹 Cleaning up memory...")

# Clear TensorFlow session
tf.keras.backend.clear_session()

# Clear CuPy memory if used
if USE_CUPY:
    mempool = cp.get_default_memory_pool()
    pinned_mempool = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned_mempool.free_all_blocks()
    print("   ✅ GPU memory cleared")

# Force garbage collection
gc.collect()

print("\n✅ Analysis complete!")
print("\n📊 Performance Summary:")
print(f"   ARIMA: Lightweight CPU implementation")
print(f"   LSTM: GPU-accelerated with mixed precision")
print(f"   GPU Usage: {'Yes' if tf.config.list_physical_devices('GPU') else 'No (CPU fallback)'}")
print(f"   CuPy Acceleration: {'Yes' if USE_CUPY else 'No (NumPy fallback)'}")

## 9. Save Results

In [ ]:
# =============================================================================
# SAVE RESULTS
# =============================================================================

# Save predictions
predictions_df = pd.DataFrame({
    'DateTime': test.index,
    'Actual': test.values,
    'ARIMA_Prediction': arima_pred_series.values,
    'LSTM_GPU_Prediction': lstm_pred_series.values
})

predictions_df.to_csv('gpu_predictions.csv', index=False)
results_df.to_csv('gpu_results.csv', index=False)

print("💾 Results saved:")
print("   - gpu_predictions.csv")
print("   - gpu_results.csv")

# Save GPU configuration
gpu_config = {
    'tensorflow_version': tf.__version__,
    'gpu_available': len(tf.config.list_physical_devices('GPU')) > 0,
    'gpu_devices': [dev.name for dev in tf.config.list_physical_devices('GPU')],
    'cupy_available': USE_CUPY,
    'mixed_precision': 'enabled' if gpus else 'disabled',
    'batch_size': BATCH_SIZE,
    'lookback': LOOKBACK
}

import json
with open('gpu_config.json', 'w') as f:
    json.dump(gpu_config, f, indent=2)

print("   - gpu_config.json")